# 🦙 Fine-Tune Llama 3.2 1B for Grammatica (VERSION 1.3 - THE FINAL FIX)

### ⚠️ CRITICAL: FIXING THE `TypeError`
The error `unexpected keyword argument 'tokenizer'` is caused by a version conflict between the newest `trl` and `unsloth`. 

**FIX:** We will use the official Unsloth installation command and a fresh environment logic.

### 🛠️ Step 0: CLEAN SLATE
Before running this, go to **Runtime > Disconnect and delete runtime**. This is essential.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.12.0" "transformers<4.46.0" "tokenizers>=0.20,<0.21" peft accelerate bitsandbytes unsloth_zoo

### 📦 Step 1: Load the 1B Model

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can handle 2048 on 1B model easily
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 📊 Step 2: Load Data (`dataset.jsonl`)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        s = ""; u = ""; a = ""
        for msg in messages:
            if msg["role"] == "system": s = msg["content"]
            elif msg["role"] == "user": u = msg["content"]
            elif msg["role"] == "assistant": a = msg["content"]
            
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{s}<|eot_id|>" \
               f"<|start_header_id|>user<|end_header_id|>\n\n{u}<|eot_id|>" \
               f"<|start_header_id|>assistant<|end_header_id|>\n\n{a}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

### 🚀 Step 3: Train!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Patch TrainingArguments if push_to_hub_token is missing (Fixes transformers v4.46+ conflict)
if not hasattr(TrainingArguments, 'push_to_hub_token'):
    TrainingArguments.push_to_hub_token = None

trainer = SFTTrainer(
    model = model, # Tokenizer is automatically handled by Unsloth-patched SFTTrainer
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 💾 Step 4: Save and Download

In [ ]:
model.save_pretrained("grammatica_lora_1b")
tokenizer.save_pretrained("grammatica_lora_1b")

!zip -r grammatica_lora.zip grammatica_lora_1b
from google.colab import files
files.download("grammatica_lora.zip")